# L6 综合实验｜一键生成可追溯月报

本实验整合第 59–60 课，用单一入口执行 Extract → Validate → Compute → Render → Archive，并生成 HTML、图表和 JSON 清单。

- 数据：固定模拟行情；非真实证券
- 输出：`outputs/L6_monthly_report.html/png/json`
- 发布边界：仅写本地文件，不发送、不上传

## 验收

- 显式报告期间和数据完整性；
- 指标、图表、HTML 共享同一结果对象；
- 同一输入重复渲染字节一致；
- 清单包含数据、报告和图表哈希；
- 异常在发布前阻断。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# 模板和哈希工具。
from jinja2 import Template
import hashlib

## 1. 单一函数返回结构化结果，而不是直接散落写文件

In [3]:
# Extract、Validate 和 Compute 形成纯数据结果。
def compute_monthly_bundle(as_of_date):
    daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
    prices = daily.pivot(index="date", columns="symbol", values="adj_close").sort_index()
    period = pd.Timestamp(as_of_date).to_period("M")
    sample = prices.loc[period.start_time:period.end_time]
    if len(sample) < 15 or sample.isna().any().any():
        raise ValueError("报告期数据未闭合或不完整")
    daily_return = sample.pct_change(fill_method=None).dropna()
    nav = sample / sample.iloc[0]
    drawdown = nav / nav.cummax() - 1
    metrics = pd.DataFrame({
        "return": sample.iloc[-1] / sample.iloc[0] - 1,
        "volatility": daily_return.std() * np.sqrt(252),
        "max_drawdown": drawdown.min(),
    })
    return {"period": str(period), "prices": sample, "nav": nav, "metrics": metrics}

source_dates = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])["date"]
bundle = compute_monthly_bundle(source_dates.max())
display(bundle["metrics"].style.format("{:.2%}"))
assert len(bundle["metrics"]) == 6

,return,volatility,max_drawdown
symbol,,,
BANK_A,6.82%,17.86%,-2.43%
ENERGY_A,-5.80%,18.28%,-6.24%
EV_A,-3.01%,20.43%,-8.31%
INSURE_A,2.65%,18.23%,-3.46%
POWER_A,0.47%,12.60%,-3.21%
TECH_A,0.10%,21.04%,-5.07%


## 2. Render 与 Archive 使用同一个 bundle

In [4]:
# 生成图表和 HTML；不重新计算指标。
axis = bundle["nav"].plot(figsize=(9, 4), title=f"L6 {bundle['period']} 月度净值（模拟数据）")
axis.set(xlabel="日期", ylabel="月初=1")
axis.grid(alpha=0.2)
axis.figure.tight_layout()
chart_path = OUTPUT_DIR / "L6_monthly_report.png"
axis.figure.savefig(chart_path, dpi=150)
plt.close(axis.figure)

template = Template('''<!doctype html><html lang="zh-CN"><meta charset="utf-8">
<title>L6 {{ period }} 月报</title><style>body{font:16px/1.6 -apple-system,"PingFang SC",sans-serif;max-width:980px;margin:40px auto;padding:0 24px}table{border-collapse:collapse;width:100%}th,td{padding:8px;border-bottom:1px solid #ddd;text-align:right}img{max-width:100%}</style>
<h1>{{ period }} 月报</h1><p>固定模拟数据，仅用于验证自动化交付。</p>
{{ table }}<img src="{{ chart }}" alt="月度净值"></html>''')
html = template.render(
    period=bundle["period"],
    table=bundle["metrics"].to_html(float_format=lambda value: f"{value:.2%}"),
    chart=chart_path.name,
)
report_path = OUTPUT_DIR / "L6_monthly_report.html"
report_path.write_text(html, encoding="utf-8")
assert report_path.exists() and chart_path.exists()

## 3. 哈希清单完成本地交付

In [5]:
# 记录输入与产物哈希，重复读取验证文件没有写坏。
data_path = DATA_DIR / "daily_prices.csv"
manifest = {
    "lab": "L6",
    "period": bundle["period"],
    "status": "ready-local",
    "data_sha256": hashlib.sha256(data_path.read_bytes()).hexdigest(),
    "report": report_path.name,
    "report_sha256": hashlib.sha256(report_path.read_bytes()).hexdigest(),
    "chart": chart_path.name,
    "chart_sha256": hashlib.sha256(chart_path.read_bytes()).hexdigest(),
}
manifest_path = OUTPUT_DIR / "L6_monthly_report.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(pd.Series(manifest, name="value").to_frame())
assert all(len(manifest[key]) == 64 for key in ["data_sha256", "report_sha256", "chart_sha256"])

,value
lab,L6
period,2025-11
status,ready-local
data_sha256,e57dd4377afd30cec702ad33494e0a98d2a5b83c8aeccc...
report,L6_monthly_report.html
report_sha256,ae84f844293311c77ca892d97b2945392a1fa6eec633b2...
chart,L6_monthly_report.png
chart_sha256,3c75d5605c02053cb165dec4dce38d7f11a2e5b201d46b...


## 方法边界与复用结论


- 自动生成不等于自动正确；数据不完整时必须在 Render 前失败。
- 本地 HTML 排版成功不替代指标单元测试和财务对账。
- 对外发布、通知接收者或写入外部系统需要另行授权。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。